## 1. Install PySpark

In [2]:
!pip install pyspark -q


## 2. Create a SparkSession

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    isnan,
    sum,
    trim,
    initcap,
    to_date,
    percentile_approx
)

In [18]:
spark = SparkSession.builder \
    .appName("SalesDataLakeAssignment") \
    .getOrCreate()

print("Spark started successfully")
print("Spark version:", spark.version)

Spark started successfully
Spark version: 4.0.4


## 3. Create the Data Lake structure

In [19]:
import os
import shutil

os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

# Copy the original CSV to Bronze
shutil.copy(
    "/content/sales.csv",
    "/content/data_lake/bronze/sales.csv"
)

print("Data Lake structure created.")

Data Lake structure created.


## 4. Read sales.csv from Bronze

In [20]:
bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Columns:")
print(df.columns)

Columns:
['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']


## 5. Display the schema

In [21]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



## 6. Display few rows

In [70]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

## 7. The total record count

In [22]:
before_count = df.count()

print("Rows before preprocessing:", before_count)

Rows before preprocessing: 1000


## 8. Use Spark operations to identify duplicates, null values, text-format inconsistencies, invalid numerical values, and inconsistent/invalid dates.

In [25]:
from pyspark.sql.functions import col, when, isnan, sum

null_report = df.select([
    sum(
        when(
            col(c).isNull(),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

null_report.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [26]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 10


## Remove exact duplicates

In [30]:
df_clean = df.dropDuplicates()

print("Rows before:", df.count())
print("Rows after duplicates removed:", df_clean.count())

Rows before: 1000
Rows after duplicates removed: 990


In [32]:
from pyspark.sql.functions import col, trim, initcap

text_columns = [
    "customer_name",
    "category",
    "city"
]

for c in text_columns:
    df_clean = df_clean.withColumn(
        c,
        initcap(trim(col(c)))
    )

In [33]:
df_clean = df_clean.withColumn(
    "quantity",
    when(col("quantity") > 0, col("quantity")).otherwise(None)
)

In [34]:
df_clean = df_clean.withColumn(
    "unit_price",
    when(col("unit_price") >= 0, col("unit_price")).otherwise(None)
)

In [35]:
df_clean = df_clean.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

In [38]:
print("Invalid discount values:")

df_clean.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show()

Invalid discount values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [39]:
print("Invalid unit price values:")

df_clean.filter(
    col("unit_price") < 0
).show()

Invalid unit price values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [40]:
print("Invalid quantity values:")

df_clean.filter(
    col("quantity") <= 0
).show()

Invalid quantity values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [52]:
from pyspark.sql.functions import (
    col, when, trim, initcap,
    try_to_timestamp, lit
)

# Start again from the original Bronze dataframe
df_clean = df.dropDuplicates()

print("Rows after removing duplicates:", df_clean.count())

Rows after removing duplicates: 990


## Convert valid dates to one consistent format and handle clearly invalid date values.

In [58]:
from pyspark.sql.functions import (
    col, trim, coalesce,
    try_to_timestamp, lit
)

df_clean = df.dropDuplicates()

df_clean = df_clean.withColumn(
    "order_date",
    coalesce(
        try_to_timestamp(
            trim(col("order_date")),
            lit("yyyy-MM-dd")
        ),
        try_to_timestamp(
            trim(col("order_date")),
            lit("yyyy/MM/dd")
        ),
        try_to_timestamp(
            trim(col("order_date")),
            lit("MM-dd-yyyy")
        ),
        try_to_timestamp(
            trim(col("order_date")),
            lit("dd/MM/yyyy")
        )
    ).cast("date")
)

In [59]:
df_clean.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [60]:
df_clean.filter(
    col("order_date").isNull()
).show(truncate=False)

+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name|product      |category |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|100097  |C0244      |Shreya Rao   |Sofa         |Furniture|4       |47430     |10              |Net Banking     |Hyderabad|Telangana    |NULL      |Returned    |
|100637  |C0020      |Aarav Patel  |Novel        |Books    |4       |610       |20              |Net Banking     |Chennai  |Tamil Nadu   |NULL      |Delivered   |
|100448  |C0270      |Pranav Iyer  |Bedside Table|Furniture|1       |4290      |35              |UPI             |Lucknow  |Uttar Pradesh|NULL      |Delivered   |
|100634  |C0262      |

In [61]:
df_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



## Show the row count before and after preprocessing.

In [62]:
after_count = df_clean.count()

print("================================")
print("VERIFICATION")
print("================================")

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)
print("Rows removed:", before_count - after_count)

VERIFICATION
Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [63]:
silver_path = "/content/data_lake/silver/preprocessed_sales"

df_clean.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver layer saved successfully.")

Silver layer saved successfully.


## Read the Silver data back with Spark and display a few rows.

In [64]:
silver_df = spark.read.parquet(
    "/content/data_lake/silver/preprocessed_sales"
)

silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [65]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



In [66]:
!find /content/data_lake -maxdepth 4 -type f

/content/data_lake/bronze/sales.csv
/content/data_lake/silver/preprocessed_sales/part-00000-0621ef06-0a4d-4fab-9eb6-984ce40fb50c-c000.snappy.parquet
/content/data_lake/silver/preprocessed_sales/._SUCCESS.crc
/content/data_lake/silver/preprocessed_sales/_SUCCESS
/content/data_lake/silver/preprocessed_sales/.part-00000-0621ef06-0a4d-4fab-9eb6-984ce40fb50c-c000.snappy.parquet.crc


In [68]:
print("==============================================")
print("       DATA LAKE ASSIGNMENT VERIFICATION")
print("==============================================")

print("\nBronze row count:")
print(before_count)

print("\nSilver row count:")
print(silver_df.count())

print("\nSilver schema:")
silver_df.printSchema()

print("\nSample Silver records:")
silver_df.show(5, truncate=False)

print("\nExact duplicate rows removed:")
print(duplicate_count)

print("\nData Lake structure:")

       DATA LAKE ASSIGNMENT VERIFICATION

Bronze row count:
1000

Silver row count:
990

Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


Sample Silver records:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state  

In [69]:
!find /content/data_lake -maxdepth 4 -type f

/content/data_lake/bronze/sales.csv
/content/data_lake/silver/preprocessed_sales/part-00000-0621ef06-0a4d-4fab-9eb6-984ce40fb50c-c000.snappy.parquet
/content/data_lake/silver/preprocessed_sales/._SUCCESS.crc
/content/data_lake/silver/preprocessed_sales/_SUCCESS
/content/data_lake/silver/preprocessed_sales/.part-00000-0621ef06-0a4d-4fab-9eb6-984ce40fb50c-c000.snappy.parquet.crc


## Preprocessing Summary

The raw `sales.csv` file was preserved unchanged in the Bronze layer.

The dataset initially contained 1,000 records and 13 columns.

The following Apache Spark preprocessing operations were performed:

1. Loaded the raw CSV file from the Bronze layer using Spark.
2. Inspected the schema, sample records, record count, missing values, and duplicate records.
3. Identified and removed 10 exact duplicate rows.
4. Trimmed unnecessary spaces from text fields.
5. Standardized capitalization inconsistencies in customer names, categories, and cities.
6. Handled missing customer names, payment methods, and cities using "Unknown".
7. Validated quantity values and converted zero/negative quantities to null before filling them with the median.
8. Validated unit prices and converted negative prices to null before filling them with the median.
9. Validated discount percentages and converted values outside the 0–100 range to null before filling them with the median.
10. Converted valid order dates to a consistent date format.
11. Clearly invalid dates were converted to null rather than deleting the complete records.
12. The processed dataset was saved in the Silver layer as Parquet.
13. The Silver data was read again using Spark for final verification.

The final Silver layer contains 990 records.

No Gold layer or advanced analysis was performed because it was not required.